# Busan Traffic AI PBL - 20_busan_streamlit_service.ipynb

부산시 교통량 예측 실습을 부산시 교통량 예측 프로젝트로 변경한 노트북입니다.
API 인증키와 ngrok token은 코드에 직접 넣지 말고 환경변수로 관리합니다.


In [ ]:
# =========================
# [셀 1] 설치
# =========================
!pip -q install streamlit pyngrok joblib scikit-learn pandas numpy matplotlib

In [ ]:
# =========================
# [셀 2] 데이터 로드 + 병합 + split
# =========================
import numpy as np
import pandas as pd

X_path = "data/processed/merged_x.csv"
y_path = "data/processed/traffic_daily_total_2022_2023.csv"

df_x = pd.read_csv(X_path)
df_y = pd.read_csv(y_path)

df_x["date"] = pd.to_datetime(df_x["date"])
df_y["date"] = pd.to_datetime(df_y["date"])

y_candidates = [c for c in df_y.columns if c != "date"]
y_col = y_candidates[0] if len(y_candidates) == 1 else ("y" if "y" in y_candidates else y_candidates[-1])

df = df_x.merge(df_y[["date", y_col]], on="date", how="inner").sort_values("date").reset_index(drop=True)

# split
train_mask = (df["date"].dt.year == 2022)
val_mask   = (df["date"].dt.year == 2023) & (df["date"].dt.month == 1)
test_mask  = (df["date"].dt.year == 2023) & (df["date"].dt.month != 1)

df_train = df[train_mask].copy()
df_val   = df[val_mask].copy()
df_test  = df[test_mask].copy()

print(df_train.shape, df_val.shape, df_test.shape)
print("y_col =", y_col)
df.head()

In [ ]:
# =========================
# [셀 3] 모델 학습 (Ridge + 전처리 파이프라인)
# =========================
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

def metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    return {"MAE": mae, "RMSE": rmse, "R2": r2}

feature_cols = [c for c in df.columns if c not in ["date", y_col]]
X_train_raw, y_train = df_train[feature_cols], df_train[y_col].values
X_val_raw,   y_val   = df_val[feature_cols],   df_val[y_col].values
X_test_raw,  y_test  = df_test[feature_cols],  df_test[y_col].values

# 숫자형만 사용(안전)
num_cols = X_train_raw.select_dtypes(include=[np.number]).columns.tolist()
X_train_raw = X_train_raw[num_cols]
X_val_raw   = X_val_raw[num_cols]
X_test_raw  = X_test_raw[num_cols]

preprocess = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

pipe = Pipeline([
    ("preprocess", preprocess),
    ("model", Ridge(alpha=1.0))
])

pipe.fit(X_train_raw, y_train)

val_pred  = pipe.predict(X_val_raw)
test_pred = pipe.predict(X_test_raw)

print("VAL :", metrics(y_val, val_pred))
print("TEST:", metrics(y_test, test_pred))

# 저장(웹앱에서 로드)
joblib.dump(pipe, "models/busan_traffic_model.joblib")
joblib.dump(num_cols, "models/feature_cols.joblib")
joblib.dump(y_col, "models/y_col.joblib")

print("saved:", "models/busan_traffic_model.joblib, models/feature_cols.joblib, models/y_col.joblib")

In [ ]:
# =========================
# [셀 4] Streamlit 앱(app.py) 만들기: 평균값 기본 입력 + 평균 대비 시각화
# =========================
app_code = r"""
import streamlit as st
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt

st.set_page_config(page_title="교통량 예측", layout="centered")

st.title("🚗 교통량 예측 웹 서비스")
st.caption("feature(X)를 하나씩 입력하면 일별 교통량 합계를 예측합니다. (기본값: 학습 데이터 평균)")

@st.cache_resource
def load_artifacts():
    model = joblib.load("models/busan_traffic_model.joblib")
    feature_cols = joblib.load("models/feature_cols.joblib")
    y_col = joblib.load("models/y_col.joblib")
    return model, feature_cols, y_col

@st.cache_data
def load_feature_means():
    df_x = pd.read_csv("data/processed/merged_x.csv")
    df_x = df_x.drop(columns=["date"], errors="ignore")
    df_x = df_x.apply(pd.to_numeric, errors="coerce")
    return df_x.mean()

@st.cache_data
def load_train_y_stats():
    # 2022년을 training으로 쓰므로, df_y에서 2022년만 평균/표준편차 계산
    df_y = pd.read_csv("data/processed/traffic_daily_total_2022_2023.csv")
    df_y["date"] = pd.to_datetime(df_y["date"])
    y_candidates = [c for c in df_y.columns if c != "date"]
    y_col = y_candidates[0] if len(y_candidates) == 1 else ("y" if "y" in y_candidates else y_candidates[-1])

    train_y = df_y[df_y["date"].dt.year == 2022][y_col].astype(float)
    return {
        "y_col": y_col,
        "mean": float(train_y.mean()),
        "std": float(train_y.std(ddof=0)),
        "p25": float(train_y.quantile(0.25)),
        "p50": float(train_y.quantile(0.50)),
        "p75": float(train_y.quantile(0.75)),
    }

pipe, feature_cols, y_col = load_artifacts()
feature_means = load_feature_means()
train_y_stats = load_train_y_stats()
train_mean = train_y_stats["mean"]
train_std  = train_y_stats["std"]

st.subheader("1) 입력 X (features)")
st.write(f"모델 타깃(y): **{y_col}**")
st.write(f"사용 feature 수: **{len(feature_cols)}**")
st.caption("각 입력칸의 기본값은 2022년 학습 데이터의 평균값입니다.")

with st.form("predict_form"):
    user_vals = {}

    for col in feature_cols:
        default_val = float(feature_means[col]) if col in feature_means and not np.isnan(feature_means[col]) else 0.0

        if col.lower() in ["is_holiday", "holiday", "isholiday"]:
            default_val = int(round(default_val))
            user_vals[col] = st.selectbox(
                f"{col} (0=평일, 1=휴일)",
                [0, 1],
                index=default_val
            )
        else:
            user_vals[col] = st.number_input(col, value=default_val)

    submitted = st.form_submit_button("예측하기")

if submitted:
    x_one = pd.DataFrame([[user_vals[c] for c in feature_cols]], columns=feature_cols)

    # 숫자형 강제 변환
    for c in x_one.columns:
        x_one[c] = pd.to_numeric(x_one[c], errors="coerce")

    pred = float(pipe.predict(x_one)[0])

    st.success(f"📈 예측 교통량(일합): {pred:,.0f}")

    # =========================
    # 2) 평균 대비 얼마나 큰가? (숫자 + 시각화)
    # =========================
    diff = pred - train_mean
    ratio = pred / train_mean if train_mean != 0 else np.nan
    z = (pred - train_mean) / train_std if train_std != 0 else np.nan  # 표준화 점수

    col1, col2, col3 = st.columns(3)
    col1.metric("2022 평균", f"{train_mean:,.0f}")
    col2.metric("예측 - 평균(Δ)", f"{diff:,.0f}", delta=f"{diff:,.0f}")
    col3.metric("예측/평균(배)", f"{ratio:.2f}x")

    st.caption(f"표준편차 기준으로는 z-score ≈ {z:.2f} (2022 학습분포 기준)")

    # 막대 그래프: 평균 vs 예측
    fig, ax = plt.subplots(figsize=(6, 4))

    labels = ["Train Mean (2022)", "Prediction"]
    values = [train_mean, pred]

    bars = ax.bar(labels, values)

    ax.set_ylabel("Daily traffic total")
    ax.set_title("Prediction vs. 2022 Training Mean", pad=15)

    # 막대 위에 값 표시 (각 막대 전용)
    for bar in bars:
        height = bar.get_height()
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            height,
            f"{height:,.0f}",
            ha="center",
            va="bottom",
            fontsize=10
        )

    # y축 상단 여유 공간 확보 (중요)
    ax.set_ylim(0, max(values) * 1.15)

    # Δ 텍스트는 막대 '사이 중앙' + 위쪽
    x_mid = 0.5  # 두 막대 사이
    y_top = max(values) * 1.08

    ax.text(
        x_mid,
        y_top,
        f"Δ {diff:,.0f}  ({ratio:.2f}x)",
        ha="center",
        va="bottom",
        fontsize=11,
        fontweight="bold",
        color="black"
    )

    st.pyplot(fig)


    # (선택) 2022 분포 분위수도 같이 보여주기
    with st.expander("2022 학습 교통량 분포(분위수) 보기"):
        st.write({
            "p25": f"{train_y_stats['p25']:,.0f}",
            "p50(median)": f"{train_y_stats['p50']:,.0f}",
            "p75": f"{train_y_stats['p75']:,.0f}",
            "mean": f"{train_mean:,.0f}",
            "std": f"{train_std:,.0f}",
        })

    with st.expander("입력값 확인"):
        st.dataframe(x_one)
"""

with open("app/streamlit_app.py", "w", encoding="utf-8") as f:
    f.write(app_code)

print("✅ app.py (평균 대비 시각화 포함) 생성 완료")
!sed -n '1,220p' app.py

In [ ]:
# =========================
# [셀 5] ngrok 설정
# =========================
from pyngrok import ngrok

import os
NGROK_TOKEN = os.getenv("NGROK_TOKEN", "")  # 환경변수 사용

if NGROK_TOKEN.strip():
    ngrok.set_auth_token(NGROK_TOKEN)
    print("✅ ngrok authtoken 설정 완료")
else:
    print("⚠️ NGROK_TOKEN이 비어 있습니다. (토큰 없이도 동작은 가능하나 제한이 있을 수 있음)")


In [ ]:
# =========================
# [셀 6] Streamlit 실행 + ngrok 터널 열기
# =========================
import subprocess
import time

# 재실행 대비: 기존 터널 종료
try:
    ngrok.kill()
except:
    pass

# Streamlit 실행 (백그라운드)
proc = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501", "--server.address", "0.0.0.0"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

time.sleep(2)

public_url = ngrok.connect(8501)
print("✅ 외부 접속 URL:", public_url.public_url)


In [ ]:
# =========================
# [셀 7] (옵션) 종료
# =========================
proc.terminate()
ngrok.kill()
print("stopped")


In [ ]:
# =========================
# [셀 1] 설치
# =========================
!pip -q install streamlit pyngrok joblib scikit-learn pandas numpy matplotlib torch

In [ ]:
# =========================
# [셀 2] 데이터 로드 + 병합 + split
# =========================
import numpy as np
import pandas as pd

X_path = "data/processed/merged_x.csv"
y_path = "data/processed/traffic_daily_total_2022_2023.csv"

df_x = pd.read_csv(X_path)
df_y = pd.read_csv(y_path)

df_x["date"] = pd.to_datetime(df_x["date"])
df_y["date"] = pd.to_datetime(df_y["date"])

y_candidates = [c for c in df_y.columns if c != "date"]
y_col = y_candidates[0] if len(y_candidates) == 1 else ("y" if "y" in y_candidates else y_candidates[-1])

df = df_x.merge(df_y[["date", y_col]], on="date", how="inner").sort_values("date").reset_index(drop=True)

# split (교수님 기준 유지)
train_mask = (df["date"].dt.year == 2022)
val_mask   = (df["date"].dt.year == 2023) & (df["date"].dt.month == 1)
test_mask  = (df["date"].dt.year == 2023) & (df["date"].dt.month != 1)

df_train = df[train_mask].copy()
df_val   = df[val_mask].copy()
df_test  = df[test_mask].copy()

print(df_train.shape, df_val.shape, df_test.shape)
print("y_col =", y_col)
df.head()


In [ ]:
# =========================
# [셀 3] MLP 학습용 전처리 + DataLoader (y 표준화 포함)
# =========================
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
import joblib

feature_cols = [c for c in df.columns if c not in ["date", y_col]]

X_train_raw, y_train = df_train[feature_cols], df_train[y_col].values.astype(np.float32)
X_val_raw,   y_val   = df_val[feature_cols],   df_val[y_col].values.astype(np.float32)
X_test_raw,  y_test  = df_test[feature_cols],  df_test[y_col].values.astype(np.float32)

# 숫자형 컬럼만 사용
num_cols = X_train_raw.select_dtypes(include=[np.number]).columns.tolist()
print("사용되는 수치형 feature:", num_cols)

X_train_raw = X_train_raw[num_cols]
X_val_raw   = X_val_raw[num_cols]
X_test_raw  = X_test_raw[num_cols]

# X 전처리
x_preprocess = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

X_train = x_preprocess.fit_transform(X_train_raw).astype(np.float32)
X_val   = x_preprocess.transform(X_val_raw).astype(np.float32)
X_test  = x_preprocess.transform(X_test_raw).astype(np.float32)

# y 표준화 (학습 안정화)
y_scaler = StandardScaler()
y_train_s = y_scaler.fit_transform(y_train.reshape(-1, 1)).astype(np.float32)
y_val_s   = y_scaler.transform(y_val.reshape(-1, 1)).astype(np.float32)
y_test_s  = y_scaler.transform(y_test.reshape(-1, 1)).astype(np.float32)

print("X:", X_train.shape, X_val.shape, X_test.shape)
print("y:", y_train_s.shape, y_val_s.shape, y_test_s.shape)

# 학습 X 평균(기본 입력값용)과 학습 y 평균(비교 기준) 저장용 계산
x_train_mean = pd.Series(X_train_raw.mean(numeric_only=True), index=num_cols)  # raw space 평균
train_y_mean = float(np.mean(y_train))
train_y_std  = float(np.std(y_train))

joblib.dump(x_preprocess, "x_preprocess.joblib")
joblib.dump(y_scaler, "y_scaler.joblib")
joblib.dump(num_cols, "models/feature_cols.joblib")
joblib.dump(y_col, "models/y_col.joblib")
joblib.dump(x_train_mean, "x_train_mean_series.joblib")
joblib.dump({"mean": train_y_mean, "std": train_y_std}, "train_y_stats.joblib")

print("saved preprocess artifacts")


In [ ]:
# =========================
# [셀 4] PyTorch 모델/학습 (val best 저장)
# =========================
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import copy

class TabDataset(Dataset):
    def __init__(self, X, y_scaled):
        self.X = torch.from_numpy(X)
        self.y = torch.from_numpy(y_scaled).view(-1, 1)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

batch_size = 64
train_loader = DataLoader(TabDataset(X_train, y_train_s), batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(TabDataset(X_val,   y_val_s),   batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(TabDataset(X_test,  y_test_s),  batch_size=batch_size, shuffle=False)

class MLPRegressor(nn.Module):
    def __init__(self, in_dim, hidden=32, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, 1)
        )

    def forward(self, x):
        return self.net(x)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = MLPRegressor(in_dim=X_train.shape[1], hidden=32, dropout=0.0).to(device)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

def eval_loader(model, loader, y_scaler):
    model.eval()
    preds_s, trues_s = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            pred = model(xb)
            preds_s.append(pred.cpu().numpy())
            trues_s.append(yb.numpy())

    preds_s = np.vstack(preds_s).reshape(-1, 1)
    trues_s = np.vstack(trues_s).reshape(-1, 1)

    preds = y_scaler.inverse_transform(preds_s).ravel()
    trues = y_scaler.inverse_transform(trues_s).ravel()

    mae  = mean_absolute_error(trues, preds)
    rmse = np.sqrt(mean_squared_error(trues, preds))
    r2   = r2_score(trues, preds)
    return mae, rmse, r2

epochs = 200

best_state = None
best_val_rmse = float("inf")
best_epoch = -1

for epoch in range(1, epochs + 1):
    model.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        pred = model(xb)
        loss = criterion(pred, yb)  # scaled space
        loss.backward()
        optimizer.step()

    if epoch % 5 == 0:
        val_mae, val_rmse, val_r2 = eval_loader(model, val_loader, y_scaler)
        print(f"epoch {epoch:03d} | val MAE {val_mae:.1f} RMSE {val_rmse:.1f} R2 {val_r2:.3f}")

        if val_rmse < best_val_rmse:
            best_val_rmse = val_rmse
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())

print(f"BEST VAL @ epoch {best_epoch:03d} | RMSE {best_val_rmse:.1f}")

# best weight 적용
model.load_state_dict(best_state)

# test 평가
test_mae, test_rmse, test_r2 = eval_loader(model, test_loader, y_scaler)
print(f"TEST (best@{best_epoch}): MAE {test_mae:.1f} RMSE {test_rmse:.1f} R2 {test_r2:.3f}")

# 모델 저장
torch.save(
    {
        "state_dict": best_state,
        "in_dim": int(X_train.shape[1]),
        "hidden": 32,
        "dropout": 0.0
    },
    "mlp_model.pt"
)

print("saved: mlp_model.pt")


In [ ]:
# =========================
# [셀 5] Streamlit 앱(app.py) 만들기: X 직접 입력 + 평균 기본값 + 평균 대비 시각화 (MLP)
# =========================
app_code = r"""
import streamlit as st
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

st.set_page_config(page_title="교통량 예측(MLP)", layout="centered")

st.title("🚗 교통량 예측 웹 서비스 (MLP)")
st.caption("feature(X)를 하나씩 입력하면 예측합니다 (기본값: 2022 학습 X 평균)")

class MLPRegressor(nn.Module):
    def __init__(self, in_dim, hidden=32, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, 1)
        )

    def forward(self, x):
        return self.net(x)

@st.cache_resource
def load_artifacts():
    x_preprocess = joblib.load("x_preprocess.joblib")
    y_scaler = joblib.load("y_scaler.joblib")
    feature_cols = joblib.load("models/feature_cols.joblib")
    y_col = joblib.load("models/y_col.joblib")
    x_train_mean = joblib.load("x_train_mean_series.joblib")  # raw space 평균
    train_y_stats = joblib.load("train_y_stats.joblib")

    ckpt = torch.load("mlp_model.pt", map_location="cpu")
    model = MLPRegressor(in_dim=ckpt["in_dim"], hidden=ckpt["hidden"], dropout=ckpt["dropout"])
    model.load_state_dict(ckpt["state_dict"])
    model.eval()

    return x_preprocess, y_scaler, feature_cols, y_col, x_train_mean, train_y_stats, model

x_preprocess, y_scaler, feature_cols, y_col, x_train_mean, train_y_stats, model = load_artifacts()

train_mean = float(train_y_stats["mean"])
train_std  = float(train_y_stats["std"])

st.subheader("1) 입력 X (features)")
st.write(f"모델 타깃(y): **{y_col}**")
st.write(f"사용 feature 수: **{len(feature_cols)}**")

with st.form("predict_form"):
    user_vals = {}

    for col in feature_cols:
        default_val = float(x_train_mean[col]) if col in x_train_mean and not np.isnan(x_train_mean[col]) else 0.0

        if col.lower() in ["is_holiday", "holiday", "isholiday"]:
            default_val = int(round(default_val))
            user_vals[col] = st.selectbox(
                f"{col} (0=평일, 1=휴일)",
                [0, 1],
                index=default_val
            )
        else:
            user_vals[col] = st.number_input(col, value=default_val)

    submitted = st.form_submit_button("예측하기")

if submitted:
    # 1행 DataFrame (raw space)
    x_one_raw = pd.DataFrame([[user_vals[c] for c in feature_cols]], columns=feature_cols)
    for c in x_one_raw.columns:
        x_one_raw[c] = pd.to_numeric(x_one_raw[c], errors="coerce")

    # 전처리 -> numpy float32
    x_one = x_preprocess.transform(x_one_raw).astype(np.float32)

    # MLP 예측 (scaled y)
    with torch.no_grad():
        xb = torch.from_numpy(x_one)
        pred_s = model(xb).numpy().reshape(-1, 1)

    # 원래 단위로 복원
    pred = float(y_scaler.inverse_transform(pred_s).ravel()[0])

    st.success(f"📈 예측 교통량(일합): {pred:,.0f}")

    # =========================
    # 2) 평균 대비 얼마나 큰가? (숫자 + 시각화)
    # =========================
    diff = pred - train_mean
    ratio = pred / train_mean if train_mean != 0 else np.nan
    z = (pred - train_mean) / train_std if train_std != 0 else np.nan

    c1, c2, c3 = st.columns(3)
    c1.metric("2022 평균", f"{train_mean:,.0f}")
    c2.metric("예측 - 평균(Δ)", f"{diff:,.0f}", delta=f"{diff:,.0f}")
    c3.metric("예측/평균(배)", f"{ratio:.2f}x")

    st.caption(f"표준편차 기준 z-score ≈ {z:.2f} (2022 기준)")

    # 막대 그래프 (겹침 방지 버전)
    fig, ax = plt.subplots(figsize=(6, 4))
    labels = ["Train Mean (2022)", "Prediction"]
    values = [train_mean, pred]
    bars = ax.bar(labels, values)

    ax.set_ylabel("Daily traffic total")
    ax.set_title("Prediction vs. 2022 Training Mean", pad=15)

    # 막대 위 값
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h, f"{h:,.0f}", ha="center", va="bottom", fontsize=10)

    ax.set_ylim(0, max(values) * 1.15)

    # Δ 텍스트: 막대 사이 중앙 + 상단
    x_mid = 0.5
    y_top = max(values) * 1.08
    ax.text(x_mid, y_top, f"Δ {diff:,.0f}  ({ratio:.2f}x)", ha="center", va="bottom",
            fontsize=11, fontweight="bold")

    st.pyplot(fig)

    with st.expander("입력값 확인"):
        st.dataframe(x_one_raw)
"""

with open("app/streamlit_app.py", "w", encoding="utf-8") as f:
    f.write(app_code)

print("✅ app.py (MLP 버전) 생성 완료")
!sed -n '1,220p' app.py


In [ ]:
# =========================
# [셀 6] ngrok 설정
# =========================
from pyngrok import ngrok

import os
NGROK_TOKEN = os.getenv("NGROK_TOKEN", "")  # 환경변수 사용

if NGROK_TOKEN.strip():
    ngrok.set_auth_token(NGROK_TOKEN)
    print("✅ ngrok authtoken 설정 완료")
else:
    print("⚠️ NGROK_TOKEN이 비어 있습니다 (토큰 없이도 될 수 있으나 제한이 있을 수 있음)")


In [ ]:
# =========================
# [셀 7] Streamlit 실행 + ngrok 터널 열기
# =========================
import subprocess
import time

try:
    ngrok.kill()
except:
    pass

proc = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501", "--server.address", "0.0.0.0"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

time.sleep(2)

public_url = ngrok.connect(8501)
print("✅ 외부 접속 URL:", public_url.public_url)

In [ ]:
# =========================
# [셀 8] (옵션) 종료
# =========================
proc.terminate()
ngrok.kill()
print("stopped")
